In [1]:
# Importation des packages et réglagle de la seed pour fixé un nombre de simulation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy_financial as npf

SEED = 2107
rng = np.random.default_rng(SEED)


In [2]:
# Paramètre de base pour les calcules

#Horizon temps
an_debut = 2027
an_fin = 2060
n_ans = an_fin - an_debut +1
cible = 3500000

#Revenue & dépense
salaire_base = 95000
Div_inc_base = 0
hypo_FIXE = 38900
fin_hypo = 2057
depense_base = 38000
fct_croissance_depense = 0.03
taux_depense_compressible = 0.12
taux_remb_dette = 0.20

#Bilan personnel au 31 Dec 2026
placement_base = 45000
propriete = 920000
inc_valeur_revente = 0
hypo_solde = 598000
autre_dette = 0

# Vérification des valeurs
valeur_nette_base = placement_base + propriete - hypo_solde - autre_dette
surplus_2026 = salaire_base - hypo_FIXE - depense_base - taux_depense_compressible * salaire_base

print("Nombre d'années:", n_ans)
print("Valeur nette:", valeur_nette_base)
print("Surplus 2026:", surplus_2026)

Nombre d'années: 34
Valeur nette: 367000
Surplus 2026: 6700.0


In [3]:
# Étape --- Hypothèque 

# Trouver le taux d'hypotèque
n_versements = fin_hypo - an_debut + 1

taux_hypo = npf.rate(nper=n_versements, pmt=-hypo_FIXE, pv=hypo_solde, fv=0)
print("taux Hypothèque :", round(taux_hypo,4))

# Faire le tableau d'amortissement
soldes = []
solde = hypo_solde
for annee in range(an_debut, an_fin + 1):
    if annee <= fin_hypo:
        interet = solde * taux_hypo
        solde = solde - (hypo_FIXE - interet)
    soldes.append(max(solde, 0))

soldes = np.array(soldes)
print("Solde en 2057 :", round(soldes[fin_hypo - an_debut],2))

taux Hypothèque : 0.0512
Solde en 2057 : 0.0


In [ ]:
# Étape 3.1.1 --- Modélisation des Taux d'intérêt (bons du trésor)

taux_base = 0.0310
n_trimestre = n_ans *4

def simuler_taux(n_scenario, rng):
    taux = np.empty((n_scenario, n_trimestre))
    r = np.full(n_scenario, taux_base)

    for t in range(n_trimestre):
        z = rng.standard_normal(n_scenario)
        r = r + 0.1 * (0.03 - r) + 0.02 * np.sqrt(np.maximum(r, 0)) * z
        taux[:, t] = np.maximum(r, 0)

    return taux

# Vérification des résultats
taux_test = simuler_taux(2500, rng)
print(taux_test.shape)
print("taux min: ", round(taux_test.min(), 4))
print("Moyenne en 2060", round(taux_test[:, -1].mean(), 4))

### Entre pour chaque Trimestre 2.9% et 3.12% selon les simulations ###

(2500, 136)
taux min:  0.0054
Moyenne en 2060 0.0298


In [ ]:
# Étape 3.2 --- Modélisation des probabilité et du salaire

variation_salaire = np.array([0.05, 0.025, 0.015, -0.02])
prob_salaire = np.array([0.10, 0.40, 0.45, 0.05])

def simuler_salaire(n_scenario, rng):
    variation = rng.choice(variation_salaire, size=(n_scenario, n_ans), p=prob_salaire)
    return salaire_base * np.cumprod(1 + variation, axis=1)
# les variation se compose à vérifié avec les mates 

# Vérification de la fonction
salaire_test = simuler_salaire(2500, rng)
print(salaire_test.shape)
print("Salaire moyen en 2027 : ", round(salaire_test[:, 0].mean()))
print("Salaire moyen en 2060 : ", round(salaire_test[:, -1].mean()))

(2500, 34)
Salaire moyen en 2027 :  96948
Salaire moyen en 2060 :  190763


In [ ]:
# Importation du fichier Données_A26
Donnees = "Données_A26.xlsx"

prix_maison = pd.read_excel(Donnees, sheet_name="PrixMaison")
taux_hebdo  = pd.read_excel(Donnees, sheet_name="TauxInteret")
salaire_qc  = pd.read_excel(Donnees, sheet_name="Salaire")
population  = pd.read_excel(Donnees, sheet_name="Population")
sp500       = pd.read_excel(Donnees, sheet_name="SP500")
success     = pd.read_excel(Donnees, sheet_name="Success")

# Vérification 
print(prix_maison)
print(prix_maison.iloc[-1,1]) # devrait être 184.25

          Mois  Indice PrixMaison
0   1999-12-31             100.00
1   2000-01-31             100.59
2   2000-02-29             101.48
3   2000-03-31             101.14
4   2000-04-30             101.59
..         ...                ...
317 2026-05-31             179.60
318 2026-06-30             183.43
319 2026-07-31             183.84
320 2026-08-31             184.23
321 2026-09-30             184.25

[322 rows x 2 columns]
184.25


In [242]:
# Étape 3.1.2 --- Modéliser le salaire hebdomadaire Moyen

# Les données dans le fichier Donnees_A26 sont mensuelle donc resample au trimestre 
salaire_qc_trim = salaire_qc.set_index("Mois")["SalaireMoyen"].resample("QE").last()

#Distribution empirique des variation Trim
variations_salaire_qc = salaire_qc_trim.pct_change().dropna().to_numpy()

salaire_qc_Hebdo_base = 1275

def simuler_salaire_qc_hebdo(n_scenario, rng):
    variation = rng.choice(variations_salaire_qc, size=(n_scenario, n_trimestre))
    return salaire_qc_Hebdo_base * np.cumprod(1 + variation, axis=1)

# vérification Donnees
print("donnees Mens:", len(salaire_qc["SalaireMoyen"]))
print("donnes trim: " , len(salaire_qc_trim))
print("variation : ", len(variations_salaire_qc))

# Vérification Variation
print("----- Dist. Variation Empirique -----")
print("nb de variation", len(variations_salaire_qc))
print("Moyenne:", round(variations_salaire_qc.mean() * 100, 4), "%")
print("Écart-type :", round(variations_salaire_qc.std(ddof=1) * 100, 4), "%")
print("Min:", round(variations_salaire_qc.min() * 100, 2))
print("max:", round(variations_salaire_qc.max() * 100, 2))

# Vérification fonction
print("----- Fonction -----")
salaire_hebdo_test = simuler_salaire_qc_hebdo(2500, rng)
print(salaire_hebdo_test.shape)
print("Salaire 2060;", round(salaire_hebdo_test[:, -1].mean()), "$")

donnees Mens: 319
donnes trim:  107
variation :  106
----- Dist. Variation Empirique -----
nb de variation 106
Moyenne: 0.694 %
Écart-type : 0.9397 %
Min: -1.7
max: 5.61
----- Fonction -----
(2500, 136)
Salaire 2060; 3259 $


In [ ]:
# Étape 3.1.3 --- Population
# Importation du PopAS_Qc_2026_sc_analyse
Population = "PopAS_Qc_2026_sc_analyse.xlsx"

# nettoyage et selection des colonnes
pop_proj = pd.read_excel(Population, sheet_name="Groupes d'âge", skiprows=7, header=None, usecols=[1, 3, 5, 6], names=['Scenario', 'Annee', 'Sexe', 'Population'])

# Selection référence (A2026) et (Sexe = 3).
pop_ref = pop_proj[(pop_proj['Scenario'] == "Référence (A2026)") & (pop_proj['Sexe'] == 3)]
pop_ref = pop_ref.set_index('Annee')['Population'].sort_index()

#Croissance 2027 à 2060
croissance_pop = pop_ref.pct_change().loc[2027 : 2060].to_numpy()

# Vérification des données
print("Nombre de croissances :", len(croissance_pop))
print("Population 2026:", int(pop_ref.loc[2026]))
print("Population 2060 :", int(pop_ref.loc[2060]))
print("Croissance moyenne:", round(croissance_pop.mean() * 100, 3), "%")
print("Min:", round(croissance_pop.min() * 100, 3), "%")
print("Max ;",round(croissance_pop.max() * 100, 3), "%")

Nombre de croissances : 34
Population 2026: 9027361
Population 2060 : 9174458
Croissance moyenne: 0.048 %
Min: -0.173 %
Max ; 0.223 %
